# Pattern 5: JWT Passthrough

The MCP server forwards the user's JWT as a Bearer token. No API key. The service validates the JWT signature via Keycloak's JWKS endpoint and extracts cryptographically proven claims.

This is the first pattern where the service verifies identity independently.

**What the service sees:** A cryptographically proven user identity with full claims.

**Weakness:** The JWT audience is broad. The token is valid for ALL services, not just this one. If the expense service is compromised, the attacker gets a token that also works against the document service.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p05_jwt_passthrough")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭────────────────────── p05_jwt_passthrough/mcp_auth.py ───────────────────────╮
│    1 """Pattern 5: JWT Passthrough.                                          │
│    2                                                                         │
│    3 The MCP server forwards the user's JWT (received from the agent caller) │
│    4 as a Bearer token. No API key. The service validates the JWT signature  │
│    5 via JWKS and extracts cryptographically proven claims.                  │
│    6                                                                         │
│    7 This is the first pattern where the service verifies identity           │
│    8 independently -- it doesn't rely on the caller being trusted.           │
│    9 """                                                                     │
│   10                                                                         │
│   11 from framework.mcp.auth import AuthHandler                              │
│   12                                                                         │
│   13                                                                         │
│   14 class JWTPassthroughHandler(AuthHandler):                               │
│   15     async def prepare_request(self, user_context, headers):             │
│   16         jwt = user_context.get("jwt")                                   │
│   17         if jwt:                                                         │
│   18             headers["Authorization"] = f"Bearer {jwt}"                  │
│   19         return headers                                                  │
│   20                                                                         │
│   21                                                                         │
│   22 auth_handler = JWTPassthroughHandler()                                  │
│   23                                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── p05_jwt_passthrough/service_auth.py ──────────────────────────────────────╮
│    1 """Pattern 5: JWT Passthrough (service side).                                                              │
│    2                                                                                                            │
│    3 The service validates the Bearer JWT signature via Keycloak's JWKS endpoint.                               │
│    4 This is the first pattern where the service verifies identity independently.                               │
│    5 No shared API key needed. The service distinguishes broad-audience tokens                                  │
│    6 (method="jwt") from narrow-audience tokens (method="scoped_jwt").                                          │
│    7 """                                                                                                        │
│    8                                                                                                            │
│    9 import jwt as pyjwt                                                                                        │
│   10 from fastapi import Request                                                                                │
│   11 from jwt import PyJWKClient                                                                                │
│   12                                                                                                            │
│   13 from framework.config import EXPECTED_ISSUER, EXPENSE_SERVICE_CLIENT_ID, DOCUMENT_SERVICE_CLIENT_ID, JWKS_ │
│   14 from framework.services.identity import Identity                                                           │
│   15                                                                                                            │
│   16 _expense_jwk_client: PyJWKClient | None = None                                                             │
│   17 _document_jwk_client: PyJWKClient | None = None                                                            │
│   18                                                                                                            │
│   19                                                                                                            │
│   20 async def get_expense_identity(request: Request) -> Identity:                                              │
│   21     return await _validate_jwt(request, EXPENSE_SERVICE_CLIENT_ID, "_expense")                             │
│   22                                                                                                            │
│   23                                                                                                            │
│   24 async def get_document_identity(request: Request) -> Identity:                                             │
│   25     return await _validate_jwt(request, DOCUMENT_SERVICE_CLIENT_ID, "_document")                           │
│   26                                                                                                            │
│   27                                                                                                            │
│   28 async def _validate_jwt(request: Request, service_client_id: str, cache_key: str) -> Identity:             │
│   29     auth_header = request.headers.get("authorization")                                                     │
│   30     if not auth_header or not auth_header.lower().startswith("bearer "):                                   │
│   31         return Identity(method="none", detail="no auth provided")                                          │
│   32                            

The MCP server just forwards the JWT. No API key, no narrowing, no OPA check.
The service validates the signature via JWKS and reads proven claims.
This is real, cryptographic identity. But look at the audience claim -- it lists multiple services.

In [3]:
await runner.start()

[04/14/26 06:32:10] INFO     StreamableHTTP session manager started                  ]8;id=10424;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=114439;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=337332;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=44913;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

[04/14/26 06:32:11] INFO     HTTP Request: POST http://127.0.0.1:50531/mcp "HTTP/1.1 200 OK"        ]8;id=35244;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=27513;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=157161;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=328580;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=837370;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=352679;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:50531/mcp "HTTP/1.1 200 OK"        ]8;id=268316;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=669625;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:50531/mcp "HTTP/1.1 200 OK"        ]8;id=649016;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=544284;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:50532/mcp "HTTP/1.1 200 OK"        ]8;id=553955;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=268736;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=864237;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=286194;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=279098;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=167099;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my recent expenses?")

                    INFO     HTTP Request: POST                                                     ]8;id=230415;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=982991;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────╮
│ [alice] What are my recent expenses? │
╰──────────────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=99870;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=711263;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=831640;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=339403;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=371957;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=660125;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:32:16] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=767317;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=293754;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:32:20] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=652959;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=293298;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "jwt", "identity_detail":    │
│     │              │      │        │ "validated JWT issued by http://localhost:8080/realms/agentauth;           │
│     │              │      │        │ aud=[\'document-service-client\', \'expense-service-client\'];             │
│     │              │      │        │ azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id": "alice",  │
│     │              │      │        │ "department": "engineering                                                 │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are your most recent expenses (4 total):                                                                   │
│                                                                                                                 │
│ - ID 1 | Department: Engineering | Category: Software | $42.50 | JetBrains AI assistant subscription | Status:  │
│ Approved                                                                                                        │
│ - ID 2 | Department: Engineering | Category: Travel | $156.00 | Train ticket to client offsite | Status:        │
│ Approved                                                                                                        │
│ - ID 3 | Department: Engineering | Category: Books | $89.00 | Designing Data-Intensive Applications | Status:   │
│ Approved                                                                                                        │
│ - ID 4 | Department: Engineering | Category: Hardware | $1,450.00 | External 4K monitor | Status: Pending       │
│                                                                                                                 │
│ Total: $1,737.50                                                                                                │
│                                                                                                                 │
│ Would you like to approve the pending item (ID 4) or see details for any item?                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your most recent expenses (4 total):\n\n- ID 1 | Department: Engineering | Category: Software | $42.50 | JetBrains AI assistant subscription | Status: Approved\n- ID 2 | Department: Engineering | Category: Travel | $156.00 | Train ticket to client offsite | Status: Approved\n- ID 3 | Department: Engineering | Category: Books | $89.00 | Designing Data-Intensive Applications | Status: Approved\n- ID 4 | Department: Engineering | Category: Hardware | $1,450.00 | External 4K monitor | Status: Pending\n\nTotal: $1,737.50\n\nWould you like to approve the pending item (ID 4) or see details for any item?', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "jwt", "identity_detail": "validated JWT issued by http://localhost:8080/realms/agentauth; aud=[\\\'document-service-client\\\', \\\'expense-service-client\\\']; azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id":

In [5]:
await runner.run_as("bob", "Show me all expenses and approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=184286;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=777377;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────────────╮
│ [bob] Show me all expenses and approve expense 4 │
╰──────────────────────────────────────────────────╯

[04/14/26 06:32:21] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=540586;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=820621;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:32:23] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=613332;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=163476;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=512915;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=672339;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:32:27] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=101518;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=806193;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:32:31] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=257620;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=88864;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses    │ {}                │      - │ {'type': 'text', 'text': '{"identity_method": "jwt",       │
│     │                 │                   │        │ "identity_detail": "validated JWT issued by                │
│     │                 │                   │        │ http://localhost:8080/realms/agentauth;                    │
│     │                 │                   │        │ aud=[\'document-service-client\',                          │
│     │                 │                   │        │ \'expense-service-client\']; azp=agent-client", "count":   │
│     │                 │                   │        │ 6, "expenses": [{"id": 1, "user_id": "alice",              │
│     │                 │                   │        │ "department": "engineering                                 │
│ 2   │ approve_expense │ {'expense_id': 4} │      - │ {'type': 'text', 'text': '{"identity_method": "jwt",       │
│     │                 │                   │        │ "approved_by": "bob", "tool_side_authz_enabled": false,    │
│     │                 │                   │        │ "expense": {"id": 4, "user_id": "alice", "department":     │
│     │                 │                   │        │ "engineering", "amount": 1450.0, "category": "hardware",   │
│     │                 │                   │        │ "description": "External 4K monitor", "status":            │
│     │                 │                   │        │ "approved"}, "_status": 200}'}                             │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are your expenses (all, no department filter applied):                                                     │
│                                                                                                                 │
│ - ID 1: user alice, department engineering, amount 42.50, category software, "JetBrains AI assistant            │
│ subscription" — approved                                                                                        │
│ - ID 2: user alice, department engineering, amount 156.00, category travel, "Train ticket to client offsite" —  │
│ approved                                                                                                        │
│ - ID 3: user alice, department engineering, amount 89.00, category books, "Designing Data-Intensive             │
│ Applications" — approved                                                                                        │
│ - ID 4: user alice, department engineering, amount 1450.00, category hardware, "External 4K monitor" — approved │
│ - ID 5: user bob, department engineering, amount 320.00, category training, "OAuth 2.0 deep-dive workshop" —    │
│ approved                                                                                                        │
│ - ID 6: user bob, department engineering, amount 67.00, category meals, "Team lunch after the migration         │
│ shipped" — approved                                                                                             │
│                                                                                                                 │
│ Note: Expense 4 (External 4K monitor) has been approved by bob. If you want to filter by department or perform  │
│ other actions, tell me which you’d like.                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your expenses (all, no department filter applied):\n\n- ID 1: user alice, department engineering, amount 42.50, category software, "JetBrains AI assistant subscription" — approved\n- ID 2: user alice, department engineering, amount 156.00, category travel, "Train ticket to client offsite" — approved\n- ID 3: user alice, department engineering, amount 89.00, category books, "Designing Data-Intensive Applications" — approved\n- ID 4: user alice, department engineering, amount 1450.00, category hardware, "External 4K monitor" — approved\n- ID 5: user bob, department engineering, amount 320.00, category training, "OAuth 2.0 deep-dive workshop" — approved\n- ID 6: user bob, department engineering, amount 67.00, category meals, "Team lunch after the migration shipped" — approved\n\nNote: Expense 4 (External 4K monitor) has been approved by bob. If you want to filter by department or perform other actions, tell me which you’d like.', tool_calls=[ToolCallTrace(nam

In [6]:
await runner.run_as("dave", "Search for compensation documents")

                    INFO     HTTP Request: POST                                                     ]8;id=265966;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=104334;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────╮
│ [dave] Search for compensation documents │
╰──────────────────────────────────────────╯

[04/14/26 06:32:32] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=490301;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=165529;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:32:35] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=345464;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=169692;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=11224;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=229953;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:32:37] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=287155;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=325727;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:32:40] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=454309;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=705523;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args                            ┃ status ┃ result                                      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': 'compensation documents'} │      - │ {'type': 'text', 'text':                    │
│     │                  │                                 │        │ '{"identity_method": "jwt",                 │
│     │                  │                                 │        │ "identity_detail": "validated JWT issued by │
│     │                  │                                 │        │ http://localhost:8080/realms/agentauth;     │
│     │                  │                                 │        │ aud=[\'document-service-client\',           │
│     │                  │                                 │        │ \'expense-service-client\'];                │
│     │                  │                                 │        │ azp=agent-client", "allowed_groups":        │
│     │                  │                                 │        │ ["admin", "platform", "public"], "count":   │
│     │                  │                                 │        │ 0, "documents": [], "                       │
└─────┴──────────────────┴─────────────────────────────────┴────────┴─────────────────────────────────────────────┘

╭──────────────────────────────────────── answer ────────────────────────────────────────╮
│ I searched for "compensation documents" but found no results within your access scope. │
│                                                                                        │
│ Want me to try broader terms or add filters? Options:                                  │
│ - q: compensation                                                                      │
│ - q: payroll                                                                           │
│ - q: salary policy                                                                     │
│ - q: compensation policy                                                               │
│ - Filter by department or date range                                                   │
│                                                                                        │
│ Tell me which terms or filters to use.                                                 │
╰────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='I searched for "compensation documents" but found no results within your access scope.\n\nWant me to try broader terms or add filters? Options:\n- q: compensation\n- q: payroll\n- q: salary policy\n- q: compensation policy\n- Filter by department or date range\n\nTell me which terms or filters to use.', tool_calls=[ToolCallTrace(name='search_documents', args={'q': 'compensation documents'}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "jwt", "identity_detail": "validated JWT issued by http://localhost:8080/realms/agentauth; aud=[\\\'document-service-client\\\', \\\'expense-service-client\\\']; azp=agent-client", "allowed_groups": ["admin", "platform", "public"], "count": 0, "documents": [], "', error=None)])

In [7]:
from framework.auth_helpers import fetch_user_jwt
from framework.display import show_token
show_token(fetch_user_jwt("alice"), label="alice's broad JWT")

                    INFO     HTTP Request: POST                                                     ]8;id=110239;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=893582;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

                            alice's broad JWT                             
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ claim              ┃ value                                             ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sub                │ 7f78567c-4e16-44d4-b723-dca7328abd88              │
│ preferred_username │ alice                                             │
│ aud                │ [document-service-client, expense-service-client] │
│ azp                │ agent-client                                      │
│ iss                │ http://localhost:8080/realms/agentauth            │
│ role               │ employee                                          │
│ department         │ engineering                                       │
│ reports_to         │ bob                                               │
│ scope              │ openid agent-claims                               │
│ exp                │ 1776175360                                        │
│ iat                │ 1776173560                                        │
│ jti                │ onrtro:f94a4fee-cbfa-9fdd-ff8b-873fac50590b       │
│ sid                │ X6ESgqCkaFkY4CSghuDRUh3G                          │
│ typ                │ Bearer                                            │
└────────────────────┴───────────────────────────────────────────────────┘

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [8]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:50529/debug/last-request "HTTP/1.1  ]8;id=765115;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=924773;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭────────────────────────────────────── expense-service /debug/last-request ──────────────────────────────────────╮
│ method:  jwt                                                                                                    │
│ user_id: bob                                                                                                    │
│ detail:  validated JWT issued by http://localhost:8080/realms/agentauth; aud=['document-service-client',        │
│ 'expense-service-client']; azp=agent-client                                                                     │
│                                                                                                                 │
│ claims:  sub=6e12d041..., role=manager, department=engineering, aud=['document-service-client',                 │
│ 'expense-service-client'], azp=agent-client                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:50530/debug/last-request "HTTP/1.1  ]8;id=418799;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=997718;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭───────────────────────────────────── document-service /debug/last-request ──────────────────────────────────────╮
│ method:  jwt                                                                                                    │
│ user_id: dave                                                                                                   │
│ detail:  validated JWT issued by http://localhost:8080/realms/agentauth; aud=['document-service-client',        │
│ 'expense-service-client']; azp=agent-client                                                                     │
│                                                                                                                 │
│ claims:  sub=bc99f071..., role=admin, department=platform, aud=['document-service-client',                      │
│ 'expense-service-client'], azp=agent-client                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The service now reports method=jwt with proven claims. It validates identity independently for the first time.

### What changed from patterns 3-4

In patterns 3-4, the service reported method=api_key -- it had no user identity and returned everything. Now the service independently verifies who the caller is via JWKS and filters accordingly. Alice sees only her expenses, bob sees all engineering expenses (he's a manager), dave sees everything (admin).

For documents, filtering is now based on proven JWT claims (department, role) rather than a hardcoded username-to-group mapping (pattern 2) or no filtering at all (patterns 1, 3-4). Dave sees admin+platform+public documents. Alice and bob see engineering+public only.

This is a fundamental shift: the service no longer trusts the caller -- it verifies identity independently.

### The broad audience problem

Look at the aud claim: it lists both `expense-service-client` and `document-service-client`. This single token is valid for every service the agent can call.

In production, blindly passing JWTs downstream creates real risk:

- **Token leakage:** if the expense service is compromised, the attacker gets a token that also works against the document service, and any other service in the audience list.
- **Audience mismatch:** services that don't strictly check the `aud` claim will accept tokens not intended for them. Our services intentionally don't enforce audience matching here to illustrate this weakness, but a production service should reject tokens where its client ID is not in the audience.
- **Lateral movement:** a compromised service can use the broad token to call other services as the user.

We need audience-scoped tokens that limit each token to a single target service.

In [9]:
await runner.stop()

Pattern p05_jwt_passthrough stopped.

[04/14/26 06:32:43] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=994466;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=728970;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       